# Data and Reference Result

**Checkpoint:** November 5, 2026  
**Team:** Chase Nielsen, Noah Champagne, and Gunnar Griffith  
**Competition:** House Prices: Advanced Regression Techniques

This notebook loads the data, checks its basic structure, and creates a simple reference result for later models.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

RANDOM_STATE = 307
DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

### Load the data

The is downloaded from the Kaggle competition. The raw files are stored locally in `data/raw/`

In [2]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

for path in [train_path, test_path, sample_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print("Training shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

train.head()

Training shape: (1460, 81)
Test shape: (1459, 80)
Sample submission shape: (1459, 2)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


### Data overview

The training data contains the target variable `SalePrice`. The test data contains the same predictor variables but do not include sale prices.

In [3]:
print("Number of predictors:", train.shape[1] - 1)
print("Target variable:", "SalePrice")
print("Duplicate training rows:", train.duplicated().sum())

column_types = train.dtypes.value_counts()
print("\nColumn types:")
print(column_types)

missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print("\nColumns with the most missing values:")
display(missing.head(15).to_frame("missing_count"))

Number of predictors: 80
Target variable: SalePrice
Duplicate training rows: 0

Column types:
object     43
int64      35
float64     3
Name: count, dtype: int64

Columns with the most missing values:


,missing_count
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageYrBlt,81
GarageCond,81
GarageType,81


### Data status

We currently have:

- `train.csv`: labeled housing data used for training and validation
- `test.csv`: unlabeled housing data used for Kaggle predictions
- `sample_submission.csv`: the required submission format
- `data_description.txt`: descriptions of the variables

The data contains both numeric and categorical predictors. Several variables have missing values. We will handle handle missing values and categorical encoding without using information from the validation set in our future pipeline.

### Local evaluation plan

We use an 80/20 training-validation split with `random_state=307`. The evaluation metric is RMSE on `log1p(SalePrice)`, which matches the Kaggle competition metric.

The split is created before fitting the model. The Kaggle test data is not used for training or local validation.

In [4]:
X = train.drop(columns="SalePrice")
y_log = np.log1p(train["SalePrice"])

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y_log,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_valid))

Training rows: 1168
Validation rows: 292


### First reference model

The first reference model predicts the mean logged sale price for every house. It doesn't use the housing predictors, but it provides a minimum performance reference for later models.

In [5]:
baseline = DummyRegressor(strategy="mean")

# Id is used to provide the required number of rows
baseline.fit(X_train[["Id"]], y_train)

valid_predictions = baseline.predict(X_valid[["Id"]])
baseline_rmse = np.sqrt(mean_squared_error(y_valid, valid_predictions))

print(f"Baseline validation log-RMSE: {baseline_rmse:.6f}")

Baseline validation log-RMSE: 0.410480


### Create a reference submission

Fit the model again using all labeled training observations. Predictions are converted back to sale-price dollars before creating a submission file.

In [7]:
baseline.fit(X[["Id"]], y_log)

test_log_predictions = baseline.predict(test[["Id"]])
test_predictions = np.expm1(test_log_predictions)

submission = sample_submission.copy()
submission["SalePrice"] = test_predictions

submission_path = OUTPUT_DIR / "baseline_submission.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,Id,SalePrice
0,1461,166716.725516
1,1462,166716.725516
2,1463,166716.725516
3,1464,166716.725516
4,1465,166716.725516


In [8]:
metrics = pd.DataFrame(
    {
        "model": ["Mean log-price reference"],
        "validation_log_rmse": [baseline_rmse],
        "training_rows": [len(X_train)],
        "validation_rows": [len(X_valid)],
    }
)

metrics_path = OUTPUT_DIR / "baseline_metrics.csv"
metrics.to_csv(metrics_path, index=False)

metrics

,model,validation_log_rmse,training_rows,validation_rows
0,Mean log-price reference,0.41048,1168,292


### Initial conclusion

The mean log-price model gives a simple reference score. A good machine learning pipeline will improve upon this validation RMSE.

Next, we will create a preprocessing pipeline for the missing values and categorical variables, then fit a Random Forest model using the same validation split and metric.